In [ ]:
"""This contains the pipeline that is needed for to run the code regarding the backtesting engine using the 
strategy mentioned 

Note:Make sure that the saved models and the scaler are all in the same files
"""

hello world


In [2]:
import os
import time
import requests
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import pywt
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


MODEL_PATH = "hybrid_cnn_lstm_gaf_model.keras"
SCALER_PATH = "gaf_scaler.pkl"

MODEL_PATH = "hybrid_cnn_lstm_gaf_model.keras"
SCALER_PATH = "gaf_scaler.pkl"
SYMBOL = "ETHUSDT"
WINDOW_SIZE = 32
WAVELET_NAME = "db4"
DECOMP_LEVEL = 1




FEATURE_COLUMNS = [
    "log_returns", "garman_klass", "log_range", 
    "volume_change", "close_open_gap", "norm_close_pos"
]


def stabilized_qlike_loss(y_true, y_pred):
    """
    Computes the standard, un-biased QLIKE loss function for volatility forecasting.
    
    Formula: L(y, y_pred) = (y / y_pred) - ln(y / y_pred) - 1.0
    
    Eliminates overprediction bias by using minimal machine-precision epsilon (1e-8)
    solely for divide-by-zero protection without distorting low-volatility ground truth.
    """
    # Cast tensors to float32 to enforce uniform precision
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Minimal machine-level epsilon strictly to prevent numerical log(0) / div-by-zero.
    # Do NOT use a custom coarse LOSS_CLIP_EPSILON here.
    eps = 1e-8
    
    y_pred_safe = tf.maximum(y_pred, eps)
    y_true_safe = tf.maximum(y_true, eps)
    
    # Compute standard QLIKE ratio and logarithmic penalty
    ratio = y_true_safe / y_pred_safe
    loss_vector = ratio - tf.math.log(ratio) - 1.0
    
    return tf.reduce_mean(loss_vector)





def get_nn_predictions(full_processed_df, steps, model, scaler, wavelet, filter_len):
    """Generates next-bar variance predictions via the Deep Learning engine."""
    start_idx = len(full_processed_df) - steps
    cnn_batch, lstm_batch, y_true = [], [], []
    
    for i in range(start_idx, len(full_processed_df)):
        slice_df = full_processed_df.iloc[:i]
        current_bar = full_processed_df.iloc[i]
        
        required_rows = WINDOW_SIZE + filter_len
        if len(slice_df) < required_rows:
            continue
            
        matrix_input = slice_df[FEATURE_COLUMNS].tail(required_rows).values
        scaled_continuous = scaler.transform(matrix_input)
        lstm_tensor = scaled_continuous[-WINDOW_SIZE:]
        
        approx_mat = np.zeros((WINDOW_SIZE, len(FEATURE_COLUMNS)))
        detail_mat = np.zeros((WINDOW_SIZE, len(FEATURE_COLUMNS)))
        
        for idx in range(len(FEATURE_COLUMNS)):
            coeffs = pywt.swt(matrix_input[:, idx], wavelet=wavelet, level=DECOMP_LEVEL, norm=True)
            approx_mat[:, idx] = coeffs[0][0][filter_len:]
            detail_mat[:, idx] = coeffs[0][1][filter_len:]
            
        def to_gasf(mat):
            min_v, max_v = mat.min(axis=0, keepdims=True), mat.max(axis=0, keepdims=True)
            denom = max_v - min_v
            denom[denom == 0] = 1.0
            scaled = -1.0 + 2.0 * ((mat - min_v) / denom)
            phi = np.arccos(np.clip(scaled, -0.999999, 0.999999))
            return np.cos(np.expand_dims(phi, axis=1) + np.expand_dims(phi, axis=0))

        gaf_image = np.concatenate([to_gasf(approx_mat), to_gasf(detail_mat)], axis=2)
        
        lstm_batch.append(lstm_tensor)
        cnn_batch.append(gaf_image)
        y_true.append(current_bar["garman_klass"])
        
    predictions = model({"cnn_input": np.array(cnn_batch), "lstm_input": np.array(lstm_batch)}, training=False)
    return predictions.numpy().flatten(), np.array(y_true)





# =====================================================================
# STABILIZED PURE HYBRID NN CONFIGURATION
# =====================================================================
PURE_NN_CONFIG = {
    # --- VOLATILITY PRE-ENTRY GATES (TIGHTENED FOR <50 TRADES) ---
    "VOL_SQUEEZE_MAX_Q": 0.25,       # Tightened from 0.30 to 0.25 (Top 25% Squeeze Only)
    "MIN_SQUEEZE_BARS": 3,           # Requires 3 bars (30 min) of continuous compression
    "EXTREME_SQUEEZE_Q": 0.10,       # Sizing boosted to 1.5x if quantile <= 10%
    "VOL_EXPANSION_EXIT_Q": 0.85,    # Peak vol take-profit exit (Top 15%)
    
    # --- MACRO TREND SLOPE GATE ---
    "MACRO_SLOPE_LOOKBACK": 36,      # 36 bars (6 hours) lookback on Macro EMA
    "MIN_MACRO_SLOPE_PCT": 0.0002,   # Very soft slope to block perfectly flat sideways markets
    
    # --- VOLUME CONFIRMATION GATE ---
    "VOLUME_MULT_THRESHOLD": 1.25,   # Tightened from 1.20x to 1.25x 24-hr SMA
    "VOLUME_SMA_WINDOW": 144,        
    
    # --- DYNAMIC TARGET & PARTIAL TP SCALING ---
    "PARTIAL_TP_PCT": 0.018,         # 50% Partial Take-Profit trigger (+1.8%)
    "PARTIAL_TP_RATIO": 0.50,        # Scale out 50% of position size
    "DYNAMIC_TP_MULT": 3.5,          # Full TP2 target = max(4.0%, 3.5 * sqrt(sigma_t+1))
    "MIN_TP_PCT": 0.040,             # Floor TP2 Target (4.0%)
    
    # --- TIMING & RISK MANAGEMENT ---
    "MIN_HOLD_BARS": 6,              # 6 bars (1 hour) min hold horizon before trailing exits
    "MOMENTUM_STALL_BARS": 18,       # NEW: Lock to breakeven if 3 hours pass without hitting Partial TP
    "MAX_HOLD_BARS": 72,             # 72 bars (12 hours) HARD TIME STOP
    "COOLDOWN_BARS": 72,             # 72 bars (12 hours) post-trade blackout window
    
    # --- STOP-LOSS & DRAWDOWN REDUCTION LOGIC ---
    "STOP_LOSS_PCT": 0.018,          # Initial 1.8% Hard Stop-Loss
    "DRAWDOWN_REDUCTION_MULT": 0.5,  # NEW: If previous trade was a loss, risk 0.5x on the next trade
    
    # --- POSITION SIZING ---
    "BASE_RISK_LONG": 1.0,           
    "BASE_RISK_SHORT": 1.0,          
    "COMPRESSION_BOOST": 1.50,       
    
    "EMA_SMOOTHING_SPAN": 18,        
    "DEFAULT_MAKER_FEE": 0.00050     # 0.05% fee per side (0.10% round trip)
}

# ==============================================================================
# 1. LIVE ENGINE PREPROCESSING LAYER
# ==============================================================================
class MarketDataStream:
    CACHE_FILENAME = "eth_usdt_5m_cache.csv"

    @staticmethod
    def _create_robust_session():
        session = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        session.mount("https://", HTTPAdapter(max_retries=retries))
        session.headers.update({"User-Agent": "Mozilla/5.0"})
        return session

    @classmethod
    def fetch_live_candles_paginated(cls, symbol, target_10m_bars):
        total_raw_needed = (target_10m_bars * 2) + 120
        if os.path.exists(cls.CACHE_FILENAME):
            try:
                df_cached = pd.read_csv(cls.CACHE_FILENAME, parse_dates=[0], index_col=0)
                df_cached.index.name = "timestamp"
                if len(df_cached) >= total_raw_needed:
                    df_cached = df_cached[~df_cached.index.duplicated(keep='first')]
                    return df_cached.iloc[-total_raw_needed:][["open", "high", "low", "close", "volume"]]
            except:
                os.remove(cls.CACHE_FILENAME)

        session = cls._create_robust_session()
        url = "https://api.binance.com/api/v3/klines"
        binance_limit_per_call = 1000
        chunks_needed = int(np.ceil(total_raw_needed / binance_limit_per_call))
        
        all_raw_candles = []
        end_time = None
        for chunk in range(chunks_needed):
            params = {"symbol": symbol, "interval": "5m", "limit": binance_limit_per_call}
            if end_time: params["endTime"] = end_time
            try:
                response = session.get(url, params=params, timeout=15)
                if response.status_code != 200: continue
                chunk_data = response.json()
                if not chunk_data: break
                all_raw_candles = chunk_data + all_raw_candles
                end_time = chunk_data[0][0] - 1
                time.sleep(0.15)
            except: time.sleep(3)
                
        df = pd.DataFrame(all_raw_candles).iloc[:, :6]
        df.columns = ["open_time", "open", "high", "low", "close", "volume"]
        df[["open", "high", "low", "close", "volume"]] = df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric)
        df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df[~df.index.duplicated(keep='first')]
        df_final = df[["open", "high", "low", "close", "volume"]]
        df_final.to_csv(cls.CACHE_FILENAME)
        return df_final

    @staticmethod
    def resample_and_engineer(df_5m):
        rules = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
        df = df_5m.resample("10min", closed="left", label="left").agg(rules).dropna()
        
        # Restore Neural Network Microstructure Features
        df["log_returns"] = np.log(df["close"] / df["close"].shift(1))
        df["garman_klass"] = 0.5 * (np.log(df["high"] / df["low"]))**2 - (2*np.log(2)-1) * (np.log(df["close"] / df["open"]))**2
        df["log_range"] = np.log(df["high"] / df["low"])
        df["volume_change"] = df["volume"].pct_change()
        df["close_open_gap"] = np.log(df["open"] / df["close"].shift(1))
        
        denom = df["high"] - df["low"]
        denom = np.where(denom == 0, 1e-8, denom)
        df["norm_close_pos"] = (df["close"] - df["low"]) / denom
        
        return df.dropna()

# ==============================================================================
# 2. STABILIZED HYBRID NN BREAKOUT ENGINE
# ==============================================================================
def run_stabilized_nn_breakout(df, raw_nn_predictions, channel_bars=60, fee_rate=0.0005, lookback_window=432, config=PURE_NN_CONFIG):
    evaluation_df = df.iloc[-len(raw_nn_predictions):].copy()
    
    # 1. SIGNAL SMOOTHING & VOLATILITY QUANTILES
    raw_pred_series = pd.Series(raw_nn_predictions, index=evaluation_df.index)
    evaluation_df["Smoothed_Predicted_Vol"] = raw_pred_series.ewm(span=config["EMA_SMOOTHING_SPAN"], adjust=False).mean()
    evaluation_df['Pred_Vol_Quantile'] = evaluation_df['Smoothed_Predicted_Vol'].rolling(window=lookback_window).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)
    
    evaluation_df['Volume_SMA'] = evaluation_df['volume'].rolling(window=config["VOLUME_SMA_WINDOW"]).mean()
    slope_lookback = config["MACRO_SLOPE_LOOKBACK"]
    evaluation_df['Macro_EMA_Slope'] = ((evaluation_df['Macro_EMA'] - evaluation_df['Macro_EMA'].shift(slope_lookback)) / evaluation_df['Macro_EMA'].shift(slope_lookback)).fillna(0.0)
    
    close = evaluation_df['close'].values
    high = evaluation_df['high'].values
    low = evaluation_df['low'].values
    volume = evaluation_df['volume'].values
    vol_sma = evaluation_df['Volume_SMA'].values
    quantiles = evaluation_df['Pred_Vol_Quantile'].values
    pred_vols = evaluation_df['Smoothed_Predicted_Vol'].values
    macro_trend = evaluation_df['Macro_Trend'].values
    macro_slope = evaluation_df['Macro_EMA_Slope'].values
    
    rolling_high = pd.Series(high, index=evaluation_df.index).shift(1).rolling(window=channel_bars).max().values
    rolling_low = pd.Series(low, index=evaluation_df.index).shift(1).rolling(window=channel_bars).min().values
    rolling_mid = (rolling_high + rolling_low) / 2.0
    
    n_bars = len(close)
    final_positions = np.zeros(n_bars)
    
    current_pos = 0.0
    entry_price = 0.0
    trade_tp_pct = 0.040
    bars_in_trade = 0
    cooldown_counter = 0
    squeeze_duration = 0
    
    breakeven_active = False
    partial_tp_taken = False
    last_trade_was_loss = False  # Track previous trade outcome for sizing
    
    trade_count = 0
    win_count = 0
    loss_count = 0
    
    for i in range(n_bars):
        q = quantiles[i]
        curr_vol = pred_vols[i]
        curr_close = close[i]
        curr_macro = macro_trend[i]
        curr_slope = macro_slope[i]
        curr_vol_mult = volume[i] / vol_sma[i] if not np.isnan(vol_sma[i]) and vol_sma[i] > 0 else 1.0
        
        if q <= config["VOL_SQUEEZE_MAX_Q"]: squeeze_duration += 1
        else: squeeze_duration = 0
        
        if current_pos != 0.0: bars_in_trade += 1
        else:
            bars_in_trade = 0
            if cooldown_counter > 0: cooldown_counter -= 1

        # --- ACTIVE POSITION RISK MANAGEMENT ---
        if current_pos != 0.0:
            unrealized_return = (curr_close - entry_price) / entry_price if current_pos > 0 else (entry_price - curr_close) / entry_price
            
            # 1. 50% PARTIAL TAKE-PROFIT TRIGGER
            if not partial_tp_taken and unrealized_return >= config["PARTIAL_TP_PCT"]:
                partial_tp_taken = True
                breakeven_active = True
                current_pos = current_pos * (1.0 - config["PARTIAL_TP_RATIO"])

            # 2. MOMENTUM STALL BREAKEVEN (Time-Decay Stop)
            if not partial_tp_taken and bars_in_trade >= config["MOMENTUM_STALL_BARS"]:
                breakeven_active = True

            effective_stop_pct = 0.000 if breakeven_active else -config["STOP_LOSS_PCT"]

            # Process Exits (Stop-Loss, Breakeven, Take-Profit, Time Stop)
            exit_triggered = False
            trade_result_is_win = False

            if unrealized_return <= effective_stop_pct:
                exit_triggered = True
                trade_result_is_win = True if partial_tp_taken else False
            elif unrealized_return >= trade_tp_pct:
                exit_triggered = True
                trade_result_is_win = True
            elif q >= config["VOL_EXPANSION_EXIT_Q"] and unrealized_return > 0.01:
                exit_triggered = True
                trade_result_is_win = True
            elif bars_in_trade >= config["MIN_HOLD_BARS"] and ((current_pos > 0 and curr_close < rolling_mid[i]) or (current_pos < 0 and curr_close > rolling_mid[i])):
                exit_triggered = True
                trade_result_is_win = True if (unrealized_return > 0 or partial_tp_taken) else False
            elif bars_in_trade >= config["MAX_HOLD_BARS"]:
                exit_triggered = True
                trade_result_is_win = True if (unrealized_return > 0 or partial_tp_taken) else False

            if exit_triggered:
                if trade_result_is_win:
                    win_count += 1
                    last_trade_was_loss = False
                else:
                    loss_count += 1
                    last_trade_was_loss = True
                
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

        # --- NEW ENTRY EVALUATION ---
        if current_pos == 0.0 and cooldown_counter == 0 and not np.isnan(rolling_high[i]) and not np.isnan(rolling_low[i]):
            
            if squeeze_duration >= config["MIN_SQUEEZE_BARS"] and curr_vol_mult >= config["VOLUME_MULT_THRESHOLD"]:
                
                # DRAWDOWN REDUCTION: Cut risk in half if previous trade was a loss
                risk_modifier = config["DRAWDOWN_REDUCTION_MULT"] if last_trade_was_loss else 1.0
                squeeze_boost = config["COMPRESSION_BOOST"] if q <= config["EXTREME_SQUEEZE_Q"] else 1.0
                size_multiplier = risk_modifier * squeeze_boost
                
                vol_std = np.sqrt(max(1e-8, curr_vol))
                trade_tp_pct = max(config["MIN_TP_PCT"], config["DYNAMIC_TP_MULT"] * vol_std)
                min_slope = config["MIN_MACRO_SLOPE_PCT"]
                
                if curr_close > rolling_high[i] and curr_macro > 0 and curr_slope >= min_slope:
                    current_pos = config["BASE_RISK_LONG"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False
                    partial_tp_taken = False
                elif curr_close < rolling_low[i] and curr_macro < 0 and curr_slope <= -min_slope:
                    current_pos = -config["BASE_RISK_SHORT"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False
                    partial_tp_taken = False

        final_positions[i] = current_pos

    # Performance Accounting
    evaluation_df['Final_Position'] = final_positions
    evaluation_df['Asset_Return'] = evaluation_df['log_returns'] if 'log_returns' in evaluation_df.columns else evaluation_df['close'].pct_change().fillna(0.0)
    evaluation_df['Raw_Strategy_Return'] = evaluation_df['Final_Position'].shift(1).fillna(0.0) * evaluation_df['Asset_Return']
    evaluation_df['Position_Change'] = evaluation_df['Final_Position'].diff().abs().fillna(0.0)
    evaluation_df['Transaction_Fees'] = evaluation_df['Position_Change'] * fee_rate
    evaluation_df['Net_Strategy_Return'] = evaluation_df['Raw_Strategy_Return'] - evaluation_df['Transaction_Fees']
    
    total_net_return = evaluation_df['Net_Strategy_Return'].sum()
    total_fee_churn = evaluation_df['Transaction_Fees'].sum()
    
    cum_returns = (1.0 + evaluation_df['Net_Strategy_Return']).cumprod() if 'log_returns' not in evaluation_df.columns else evaluation_df['Net_Strategy_Return'].cumsum().apply(np.exp)
    running_max = cum_returns.cummax()
    max_drawdown = ((cum_returns - running_max) / running_max).min()
    
    win_rate = (win_count / trade_count * 100.0) if trade_count > 0 else 0.0
    
    return total_net_return, max_drawdown, total_fee_churn, trade_count, win_rate, evaluation_df


# ==============================================================================
# 3. METRICS CALCULATOR & MAIN SYSTEM RUNNER
# ==============================================================================
def compute_advanced_metrics(evaluation_df):
    net_returns = evaluation_df['Net_Strategy_Return']
    daily_returns = net_returns.resample('D').sum()
    mean_daily = daily_returns.mean()
    std_daily = daily_returns.std()
    
    sharpe_ratio = (mean_daily / (std_daily + 1e-8)) * np.sqrt(365) if std_daily > 0 else 0.0
    
    gross_profits = net_returns[net_returns > 0].sum()
    gross_losses = abs(net_returns[net_returns < 0].sum())
    profit_factor = (gross_profits / gross_losses) if gross_losses > 0 else np.inf
    
    monthly_returns = net_returns.groupby(pd.Grouper(freq='MS')).sum()
    return sharpe_ratio, profit_factor, monthly_returns


if __name__ == "__main__":
    TEST_BARS = 25550   
    FEE_RATE = 0.0005   # 0.05% per side
    OPTIMAL_CHANNEL_BARS = 60  
    
    nn_model = tf.keras.models.load_model(MODEL_PATH, custom_objects={"stabilized_qlike_loss": stabilized_qlike_loss})
    nn_scaler = joblib.load(SCALER_PATH)
    wavelet = pywt.Wavelet(WAVELET_NAME)
    
    df_raw = MarketDataStream.fetch_live_candles_paginated(SYMBOL, target_10m_bars=TEST_BARS)
    df_features = MarketDataStream.resample_and_engineer(df_raw)
    
    price_series = df_features["close"] if "close" in df_features.columns else df_features["norm_close_pos"]
    df_features["Macro_EMA"] = price_series.ewm(span=1200, adjust=False).mean()
    df_features["Macro_Trend"] = np.where(price_series >= df_features["Macro_EMA"], 1.0, -1.0)

    y_pred_nn, y_true = get_nn_predictions(df_features, TEST_BARS, nn_model, nn_scaler, wavelet, wavelet.dec_len)
    
    ret, dd, fees, trades, win_rate, eval_df = run_stabilized_nn_breakout(
        df_features, y_pred_nn, channel_bars=OPTIMAL_CHANNEL_BARS, fee_rate=FEE_RATE
    )
    
    sharpe_ratio, profit_factor, monthly_returns = compute_advanced_metrics(eval_df)

    print("\n" + "="*65)
    print("   STABILIZED 'ANTI-CHOP' NN BREAKOUT PERFORMANCE REPORT")
    print("="*65)
    print(f"Total Net Return            : {ret*100:+.2f}%")
    print(f"Max Peak-to-Trough Drawdown : {dd*100:.2f}%")
    print(f"Annualized Sharpe Ratio     : {sharpe_ratio:.2f}")
    print(f"Strategy Profit Factor      : {profit_factor:.2f}")
    print(f"Win Rate                    : {win_rate:.1f}%")
    print(f"Total Trades Executed       : {trades}")
    print(f"Total Exchange Fees Paid    : {fees*100:.2f}%")
    print("-"*65)
    print("   MONTH-BY-MONTH NET RETURN BREAKDOWN")
    print("-"*65)
    for month_dt, m_ret in monthly_returns.items():
        month_str = month_dt.strftime('%b %Y')
        status = "🟢 PROFIT" if m_ret > 0 else ("🔴 LOSS" if m_ret < 0 else "⚪ NEUTRAL")
        print(f"{month_str:<12} | {m_ret*100:>+10.2f}% | {status}")
    print("="*65)

C:\Users\dreal\AppData\Local\Temp\ipykernel_16304\1424543602.py:237: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  evaluation_df['Pred_Vol_Quantile'] = evaluation_df['Smoothed_Predicted_Vol'].rolling(window=lookback_window).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)



   STABILIZED 'ANTI-CHOP' NN BREAKOUT PERFORMANCE REPORT
Total Net Return            : +5.43%
Max Peak-to-Trough Drawdown : -3.95%
Annualized Sharpe Ratio     : 1.04
Strategy Profit Factor      : 1.09
Win Rate                    : 38.6%
Total Trades Executed       : 44
Total Exchange Fees Paid    : 3.85%
-----------------------------------------------------------------
   MONTH-BY-MONTH NET RETURN BREAKDOWN
-----------------------------------------------------------------
Jan 2026     |      +0.00% | ⚪ NEUTRAL
Feb 2026     |      -1.81% | 🔴 LOSS
Mar 2026     |      -0.06% | 🔴 LOSS
Apr 2026     |      +3.01% | 🟢 PROFIT
May 2026     |      +0.11% | 🟢 PROFIT
Jun 2026     |      -2.98% | 🔴 LOSS
Jul 2026     |      +7.16% | 🟢 PROFIT


10-minute ETH breakout momentum system modeled on 0.02% Post-Only Maker limit order execution to minimize exchange fee drag across a 60-bar (10-hour) Donchian Channel. Entries trigger on candle-close breakouts conditioned on 200-hour Macro EMA trend and slope alignment, volume expansion (≥1.2x 24-hour SMA), and Neural Network forward volatility ranking in the bottom 30th percentile (boosted to 1.5x leverage if compressed below the 15th percentile). Active trade management utilizes an initial 1.8% hard stop, an automated 50% partial scale-out at +1.8% that shifts the remaining stop to breakeven (0.0%), a dynamic volatility-scaled runner target (max[4.0%, 3.5 * sqrt(sigma)]), an exhaustion exit at the 85th volatility percentile, a trailing channel-midpoint stop after a 1-hour minimum hold, and a hard 12-hour time stop followed by a 12-hour cooldown.


In [3]:
import os
import time
import requests
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import pywt
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

def log(msg):
    print(f"[LOG {time.strftime('%H:%M:%S')}] {msg}", flush=True)

# =====================================================================
# CLEAN BASELINE CONFIGURATION (POST-ONLY MAKER LIMIT FEES)
# =====================================================================
PURE_NN_CONFIG = {
    # --- VOLATILITY PRE-ENTRY GATES ---
    "VOL_SQUEEZE_MAX_Q": 0.30,       # Bottom 30th percentile volatility compression
    "MIN_SQUEEZE_BARS": 3,           # Requires 3 bars (30 min) of continuous compression
    "EXTREME_SQUEEZE_Q": 0.15,       # Sizing boosted to 1.5x if quantile <= 15%
    "VOL_EXPANSION_EXIT_Q": 0.85,    # Peak vol take-profit exit (Top 15%)
    
    # --- MACRO TREND SLOPE GATE ---
    "MACRO_SLOPE_LOOKBACK": 36,      # 36 bars (6 hours) lookback on Macro EMA
    "MIN_MACRO_SLOPE_PCT": 0.0004,   # Soft slope threshold to filter flat sideways chop
    
    # --- VOLUME CONFIRMATION GATE ---
    "VOLUME_MULT_THRESHOLD": 1.20,   # Volume >= 1.20x 24-hr SMA
    "VOLUME_SMA_WINDOW": 144,        # 144 10m bars = 24 hours
    
    # --- DYNAMIC TARGET & PARTIAL TP SCALING ---
    "PARTIAL_TP_PCT": 0.018,         # 50% Partial Take-Profit trigger (+1.8%)
    "PARTIAL_TP_RATIO": 0.50,        # Scale out 50% of position size
    "DYNAMIC_TP_MULT": 3.5,          # Full TP2 target = max(4.0%, 3.5 * sqrt(sigma_t+1))
    "MIN_TP_PCT": 0.040,             # Floor TP2 Target (4.0%)
    
    # --- TIMING & RISK MANAGEMENT ---
    "MIN_HOLD_BARS": 6,              # 6 bars (1 hour) min hold horizon before trailing exits
    "MAX_HOLD_BARS": 72,             # 72 bars (12 hours) HARD TIME STOP
    "COOLDOWN_BARS": 72,             # 72 bars (12 hours) post-trade blackout window
    
    # --- STOP-LOSS LOGIC ---
    "STOP_LOSS_PCT": 0.018,          # Initial 1.8% Hard Stop-Loss
    
    # --- POSITION SIZING ---
    "BASE_RISK_LONG": 1.0,           
    "BASE_RISK_SHORT": 1.0,          
    "COMPRESSION_BOOST": 1.50,       
    
    "EMA_SMOOTHING_SPAN": 18,        
    "DEFAULT_MAKER_FEE": 0.00020     # 0.02% Post-Only Maker Fee per side (0.04% round trip)
}

# ==============================================================================
# 1. LIVE ENGINE PREPROCESSING LAYER
# ==============================================================================
class MarketDataStream:
    CACHE_FILENAME = "eth_usdt_5m_cache.csv"

    @staticmethod
    def _create_robust_session():
        session = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        session.mount("https://", HTTPAdapter(max_retries=retries))
        session.headers.update({"User-Agent": "Mozilla/5.0"})
        return session

    @classmethod
    def fetch_live_candles_paginated(cls, symbol, target_10m_bars):
        total_raw_needed = (target_10m_bars * 2) + 120
        
        if os.path.exists(cls.CACHE_FILENAME):
            log(f"Found local cache file '{cls.CACHE_FILENAME}'. Validating size...")
            try:
                df_cached = pd.read_csv(cls.CACHE_FILENAME, parse_dates=[0], index_col=0)
                df_cached.index.name = "timestamp"
                if len(df_cached) >= total_raw_needed:
                    log(f"Cache successfully loaded with {len(df_cached)} 5m rows. Skipping API call.")
                    df_cached = df_cached[~df_cached.index.duplicated(keep='first')]
                    return df_cached.iloc[-total_raw_needed:][["open", "high", "low", "close", "volume"]]
                else:
                    log(f"Cache too small ({len(df_cached)} < {total_raw_needed}). Fetching from Binance...")
            except Exception as e:
                log(f"Cache read failed ({e}). Re-downloading data...")
                os.remove(cls.CACHE_FILENAME)

        session = cls._create_robust_session()
        url = "https://api.binance.com/api/v3/klines"
        binance_limit_per_call = 1000
        chunks_needed = int(np.ceil(total_raw_needed / binance_limit_per_call))
        
        log(f"Fetching {total_raw_needed} raw candles from Binance over {chunks_needed} paginated requests...")
        all_raw_candles = []
        end_time = None
        
        for chunk in range(chunks_needed):
            params = {"symbol": symbol, "interval": "5m", "limit": binance_limit_per_call}
            if end_time: params["endTime"] = end_time
            try:
                response = session.get(url, params=params, timeout=15)
                if response.status_code != 200:
                    time.sleep(1)
                    continue
                chunk_data = response.json()
                if not chunk_data: break
                all_raw_candles = chunk_data + all_raw_candles
                end_time = chunk_data[0][0] - 1
                if (chunk + 1) % 5 == 0 or (chunk + 1) == chunks_needed:
                    log(f"Downloaded API Chunk {chunk + 1}/{chunks_needed}...")
                time.sleep(0.1)
            except Exception as e:
                log(f"API Chunk warning ({e}). Retrying...")
                time.sleep(2)
                
        df = pd.DataFrame(all_raw_candles).iloc[:, :6]
        df.columns = ["open_time", "open", "high", "low", "close", "volume"]
        df[["open", "high", "low", "close", "volume"]] = df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric)
        df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df[~df.index.duplicated(keep='first')]
        df_final = df[["open", "high", "low", "close", "volume"]]
        df_final.to_csv(cls.CACHE_FILENAME)
        log(f"Saved {len(df_final)} raw candles to local cache '{cls.CACHE_FILENAME}'.")
        return df_final

    @staticmethod
    def resample_and_engineer(df_5m):
        log("Resampling 5m candles to 10m timeframe & computing feature engineering...")
        rules = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
        df = df_5m.resample("10min", closed="left", label="left").agg(rules).dropna()
        
        # Microstructure Features
        df["log_returns"] = np.log(df["close"] / df["close"].shift(1))
        df["garman_klass"] = 0.5 * (np.log(df["high"] / df["low"]))**2 - (2*np.log(2)-1) * (np.log(df["close"] / df["open"]))**2
        df["log_range"] = np.log(df["high"] / df["low"])
        df["volume_change"] = df["volume"].pct_change()
        df["close_open_gap"] = np.log(df["open"] / df["close"].shift(1))
        
        denom = df["high"] - df["low"]
        denom = np.where(denom == 0, 1e-8, denom)
        df["norm_close_pos"] = (df["close"] - df["low"]) / denom
        
        log("Feature engineering complete!")
        return df.dropna()

# ==============================================================================
# 2. CLEAN BASELINE HYBRID NN BREAKOUT BACKTEST ENGINE
# ==============================================================================
def run_pure_nn_breakout_backtest(df, raw_nn_predictions, channel_bars=60, fee_rate=0.0002, lookback_window=432, config=PURE_NN_CONFIG):
    log("Executing Strategy Backtest Loop with Post-Only Maker Limit Fees (0.02%)...")
    evaluation_df = df.iloc[-len(raw_nn_predictions):].copy()
    
    # 1. SIGNAL SMOOTHING & VOLATILITY QUANTILES
    raw_pred_series = pd.Series(raw_nn_predictions, index=evaluation_df.index)
    evaluation_df["Smoothed_Predicted_Vol"] = raw_pred_series.ewm(span=config["EMA_SMOOTHING_SPAN"], adjust=False).mean()
    evaluation_df['Pred_Vol_Quantile'] = evaluation_df['Smoothed_Predicted_Vol'].rolling(window=lookback_window).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)
    
    evaluation_df['Volume_SMA'] = evaluation_df['volume'].rolling(window=config["VOLUME_SMA_WINDOW"]).mean()
    slope_lookback = config["MACRO_SLOPE_LOOKBACK"]
    evaluation_df['Macro_EMA_Slope'] = ((evaluation_df['Macro_EMA'] - evaluation_df['Macro_EMA'].shift(slope_lookback)) / evaluation_df['Macro_EMA'].shift(slope_lookback)).fillna(0.0)
    
    close = evaluation_df['close'].values
    high = evaluation_df['high'].values
    low = evaluation_df['low'].values
    volume = evaluation_df['volume'].values
    vol_sma = evaluation_df['Volume_SMA'].values
    quantiles = evaluation_df['Pred_Vol_Quantile'].values
    pred_vols = evaluation_df['Smoothed_Predicted_Vol'].values
    macro_trend = evaluation_df['Macro_Trend'].values
    macro_slope = evaluation_df['Macro_EMA_Slope'].values
    
    rolling_high = pd.Series(high, index=evaluation_df.index).shift(1).rolling(window=channel_bars).max().values
    rolling_low = pd.Series(low, index=evaluation_df.index).shift(1).rolling(window=channel_bars).min().values
    rolling_mid = (rolling_high + rolling_low) / 2.0
    
    n_bars = len(close)
    final_positions = np.zeros(n_bars)
    
    current_pos = 0.0
    entry_price = 0.0
    trade_tp_pct = 0.040
    bars_in_trade = 0
    cooldown_counter = 0
    squeeze_duration = 0
    breakeven_active = False
    partial_tp_taken = False
    
    trade_count = 0
    win_count = 0
    loss_count = 0
    
    for i in range(n_bars):
        q = quantiles[i]
        curr_vol = pred_vols[i]
        curr_close = close[i]
        curr_macro = macro_trend[i]
        curr_slope = macro_slope[i]
        curr_vol_mult = volume[i] / vol_sma[i] if not np.isnan(vol_sma[i]) and vol_sma[i] > 0 else 1.0
        
        if q <= config["VOL_SQUEEZE_MAX_Q"]: squeeze_duration += 1
        else: squeeze_duration = 0
        
        if current_pos != 0.0: bars_in_trade += 1
        else:
            bars_in_trade = 0
            if cooldown_counter > 0: cooldown_counter -= 1

        # --- ACTIVE POSITION RISK MANAGEMENT ---
        if current_pos != 0.0:
            unrealized_return = (curr_close - entry_price) / entry_price if current_pos > 0 else (entry_price - curr_close) / entry_price
            
            # 1. 50% PARTIAL TAKE-PROFIT TRIGGER (+1.8%)
            if not partial_tp_taken and unrealized_return >= config["PARTIAL_TP_PCT"]:
                partial_tp_taken = True
                breakeven_active = True
                current_pos = current_pos * (1.0 - config["PARTIAL_TP_RATIO"])

            effective_stop_pct = 0.000 if breakeven_active else -config["STOP_LOSS_PCT"]

            # 2. Stop-Loss / Breakeven Exit
            if unrealized_return <= effective_stop_pct:
                if effective_stop_pct < 0:
                    loss_count += 1
                else:
                    if partial_tp_taken: win_count += 1
                    else: loss_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue
                
            # 3. Dynamic TP2 Exit
            if unrealized_return >= trade_tp_pct:
                win_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

            # 4. Vol Expansion Peak Exit
            if q >= config["VOL_EXPANSION_EXIT_Q"] and unrealized_return > 0.01:
                win_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

            # 5. Trailing Midpoint Exit
            if bars_in_trade >= config["MIN_HOLD_BARS"]:
                if (current_pos > 0 and curr_close < rolling_mid[i]) or (current_pos < 0 and curr_close > rolling_mid[i]):
                    if unrealized_return > 0 or partial_tp_taken: win_count += 1
                    else: loss_count += 1
                    current_pos = 0.0
                    trade_count += 1
                    cooldown_counter = config["COOLDOWN_BARS"]
                    final_positions[i] = 0.0
                    continue

            # 6. Hard Time Stop (12 Hours)
            if bars_in_trade >= config["MAX_HOLD_BARS"]:
                if unrealized_return > 0 or partial_tp_taken: win_count += 1
                else: loss_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

        # --- NEW ENTRY EVALUATION ---
        if current_pos == 0.0 and cooldown_counter == 0 and not np.isnan(rolling_high[i]) and not np.isnan(rolling_low[i]):
            if squeeze_duration >= config["MIN_SQUEEZE_BARS"] and curr_vol_mult >= config["VOLUME_MULT_THRESHOLD"]:
                size_multiplier = config["COMPRESSION_BOOST"] if q <= config["EXTREME_SQUEEZE_Q"] else 1.0
                vol_std = np.sqrt(max(1e-8, curr_vol))
                trade_tp_pct = max(config["MIN_TP_PCT"], config["DYNAMIC_TP_MULT"] * vol_std)
                min_slope = config["MIN_MACRO_SLOPE_PCT"]
                
                # Long Breakout
                if curr_close > rolling_high[i] and curr_macro > 0 and curr_slope >= min_slope:
                    current_pos = config["BASE_RISK_LONG"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False
                    partial_tp_taken = False

                # Short Breakout
                elif curr_close < rolling_low[i] and curr_macro < 0 and curr_slope <= -min_slope:
                    current_pos = -config["BASE_RISK_SHORT"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False
                    partial_tp_taken = False

        final_positions[i] = current_pos

    evaluation_df['Final_Position'] = final_positions
    evaluation_df['Asset_Return'] = evaluation_df['log_returns'] if 'log_returns' in evaluation_df.columns else evaluation_df['close'].pct_change().fillna(0.0)
    evaluation_df['Raw_Strategy_Return'] = evaluation_df['Final_Position'].shift(1).fillna(0.0) * evaluation_df['Asset_Return']
    evaluation_df['Position_Change'] = evaluation_df['Final_Position'].diff().abs().fillna(0.0)
    evaluation_df['Transaction_Fees'] = evaluation_df['Position_Change'] * fee_rate
    evaluation_df['Net_Strategy_Return'] = evaluation_df['Raw_Strategy_Return'] - evaluation_df['Transaction_Fees']
    
    total_net_return = evaluation_df['Net_Strategy_Return'].sum()
    total_fee_churn = evaluation_df['Transaction_Fees'].sum()
    
    cum_returns = (1.0 + evaluation_df['Net_Strategy_Return']).cumprod() if 'log_returns' not in evaluation_df.columns else evaluation_df['Net_Strategy_Return'].cumsum().apply(np.exp)
    running_max = cum_returns.cummax()
    max_drawdown = ((cum_returns - running_max) / running_max).min()
    
    win_rate = (win_count / trade_count * 100.0) if trade_count > 0 else 0.0
    log("Strategy backtest execution complete!")
    return total_net_return, max_drawdown, total_fee_churn, trade_count, win_rate, evaluation_df

# ==============================================================================
# 3. METRICS CALCULATOR & MAIN SYSTEM RUNNER
# ==============================================================================
def compute_advanced_metrics(evaluation_df):
    net_returns = evaluation_df['Net_Strategy_Return']
    daily_returns = net_returns.resample('D').sum()
    mean_daily = daily_returns.mean()
    std_daily = daily_returns.std()
    
    sharpe_ratio = (mean_daily / (std_daily + 1e-8)) * np.sqrt(365) if std_daily > 0 else 0.0
    
    gross_profits = net_returns[net_returns > 0].sum()
    gross_losses = abs(net_returns[net_returns < 0].sum())
    profit_factor = (gross_profits / gross_losses) if gross_losses > 0 else np.inf
    
    monthly_returns = net_returns.groupby(pd.Grouper(freq='MS')).sum()
    return sharpe_ratio, profit_factor, monthly_returns


if __name__ == "__main__":
    log("Starting Clean Baseline Strategy Engine with Post-Only Maker Fees...")
    
    TEST_BARS = 25550   
    FEE_RATE = 0.0002   # 0.02% per side Post-Only Maker Fee (0.04% round trip)
    OPTIMAL_CHANNEL_BARS = 60  
    
    log("Loading Neural Network & Scaler assets...")
    nn_model = tf.keras.models.load_model(MODEL_PATH, custom_objects={"stabilized_qlike_loss": stabilized_qlike_loss})
    nn_scaler = joblib.load(SCALER_PATH)
    wavelet = pywt.Wavelet(WAVELET_NAME)
    
    df_raw = MarketDataStream.fetch_live_candles_paginated(SYMBOL, target_10m_bars=TEST_BARS)
    df_features = MarketDataStream.resample_and_engineer(df_raw)
    
    price_series = df_features["close"] if "close" in df_features.columns else df_features["norm_close_pos"]
    df_features["Macro_EMA"] = price_series.ewm(span=1200, adjust=False).mean()
    df_features["Macro_Trend"] = np.where(price_series >= df_features["Macro_EMA"], 1.0, -1.0)

    y_pred_nn, y_true = get_nn_predictions(df_features, TEST_BARS, nn_model, nn_scaler, wavelet, wavelet.dec_len)
    
    ret, dd, fees, trades, win_rate, eval_df = run_pure_nn_breakout_backtest(
        df_features, y_pred_nn, channel_bars=OPTIMAL_CHANNEL_BARS, fee_rate=FEE_RATE
    )
    
    sharpe_ratio, profit_factor, monthly_returns = compute_advanced_metrics(eval_df)

    print("\n" + "="*65, flush=True)
    print("   CLEAN BASELINE (0.02% MAKER FEE) PERFORMANCE REPORT", flush=True)
    print("="*65, flush=True)
    print(f"Total Net Return            : {ret*100:+.2f}%", flush=True)
    print(f"Max Peak-to-Trough Drawdown : {dd*100:.2f}%", flush=True)
    print(f"Annualized Sharpe Ratio     : {sharpe_ratio:.2f}", flush=True)
    print(f"Strategy Profit Factor      : {profit_factor:.2f}", flush=True)
    print(f"Win Rate                    : {win_rate:.1f}%", flush=True)
    print(f"Total Trades Executed       : {trades}", flush=True)
    print(f"Total Exchange Fees Paid    : {fees*100:.2f}%", flush=True)
    print("-"*65, flush=True)
    print("   MONTH-BY-MONTH NET RETURN BREAKDOWN", flush=True)
    print("-"*65, flush=True)
    for month_dt, m_ret in monthly_returns.items():
        month_str = month_dt.strftime('%b %Y')
        status = "🟢 PROFIT" if m_ret > 0 else ("🔴 LOSS" if m_ret < 0 else "⚪ NEUTRAL")
        print(f"{month_str:<12} | {m_ret*100:>+10.2f}% | {status}", flush=True)
    print("="*65, flush=True)

[LOG 16:56:45] Starting Clean Baseline Strategy Engine with Post-Only Maker Fees...
[LOG 16:56:45] Loading Neural Network & Scaler assets...
[LOG 16:56:45] Found local cache file 'eth_usdt_5m_cache.csv'. Validating size...
[LOG 16:56:45] Cache successfully loaded with 63000 5m rows. Skipping API call.
[LOG 16:56:45] Resampling 5m candles to 10m timeframe & computing feature engineering...
[LOG 16:56:45] Feature engineering complete!
[LOG 16:58:20] Executing Strategy Backtest Loop with Post-Only Maker Limit Fees (0.02%)...


C:\Users\dreal\AppData\Local\Temp\ipykernel_16304\4085462163.py:160: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  evaluation_df['Pred_Vol_Quantile'] = evaluation_df['Smoothed_Predicted_Vol'].rolling(window=lookback_window).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)


[LOG 16:58:23] Strategy backtest execution complete!

   CLEAN BASELINE (0.02% MAKER FEE) PERFORMANCE REPORT
Total Net Return            : +12.86%
Max Peak-to-Trough Drawdown : -7.94%
Annualized Sharpe Ratio     : 1.76
Strategy Profit Factor      : 1.11
Win Rate                    : 43.2%
Total Trades Executed       : 44
Total Exchange Fees Paid    : 2.23%
-----------------------------------------------------------------
   MONTH-BY-MONTH NET RETURN BREAKDOWN
-----------------------------------------------------------------
Jan 2026     |      +0.00% | ⚪ NEUTRAL
Feb 2026     |      -3.41% | 🔴 LOSS
Mar 2026     |      +3.74% | 🟢 PROFIT
Apr 2026     |      +3.56% | 🟢 PROFIT
May 2026     |      +2.70% | 🟢 PROFIT
Jun 2026     |      -6.43% | 🔴 LOSS
Jul 2026     |     +12.70% | 🟢 PROFIT


In [5]:
def calculate_qlike(y_true, y_pred):
    y_true_stab = np.clip(y_true, 1e-8, None)
    y_pred_stab = np.clip(y_pred, 1e-8, None)
    ratio = y_true_stab / y_pred_stab
    return ratio - np.log(ratio) - 1.0


This strategy is an institutional-grade, volatility-gated momentum breakout system that executes 10-minute bar closes outside a 60-bar (10-hour) Donchian Channel, filtered to enter exclusively during pre-expansion consolidation. Entries require directional alignment with a 200-hour Macro EMA, volume confirmation (≥1.2x 24-hour SMA), and a Hybrid Neural Network volatility forecast ranking in the bottom 30th percentile (q ≤ 0.30, boosted to 1.5x size for extreme compression at q ≤ 0.15). Positions are managed through multi-tiered exits including a breakeven ratchet, a dynamic volatility-scaled profit target (max[4.0%, 3.5 * sqrt(sigma)]), a volatility exhaustion exit (q ≥ 0.85), a trailing Donchian midpoint stop after a 1-hour minimum hold, and a hard 12-hour time stop enforced with a mandatory 12-hour post-trade cooldown.

In [ ]:

import os
import time
import requests
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import pywt
from statsmodels.tsa.stattools import adfuller
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# =====================================================================
# PURE HYBRID NN BREAKOUT CONFIGURATION
# =====================================================================
PURE_NN_CONFIG = {
    # --- VOLATILITY PRE-ENTRY GATES ---
    "VOL_SQUEEZE_MAX_Q": 0.30,       # Breakouts allowed ONLY if forecasted vol quantile <= 30%
    "EXTREME_SQUEEZE_Q": 0.15,       # Sizing boosted to 1.5x if quantile <= 15%
    "VOL_EXPANSION_EXIT_Q": 0.85,    # Peak vol take-profit exit (Top 15%)
    
    # --- VOLUME CONFIRMATION GATE ---
    "VOLUME_MULT_THRESHOLD": 1.20,   # Volume must be >= 1.2x 24-hr SMA
    "VOLUME_SMA_WINDOW": 144,        # 144 10m bars = 24 hours
    
    # --- DYNAMIC TARGET SCALING ---
    "DYNAMIC_TP_MULT": 3.5,          # TP = max(4.0%, 3.5 * sqrt(sigma_t+1))
    "MIN_TP_PCT": 0.040,             # Floor Take-Profit (4.0%)
    
    # --- TIMING & RISK MANAGEMENT ---
    "MIN_HOLD_BARS": 6,              # 6 bars (1 hour) min hold horizon
    "MAX_HOLD_BARS": 72,             # 72 bars (12 hours) HARD TIME STOP
    "COOLDOWN_BARS": 72,             # 72 bars (12 hours) post-trade blackout window
    
    # --- STOP-LOSS & BREAKEVEN LOGIC ---
    "STOP_LOSS_PCT": 0.018,          # 1.8% Hard Stop-Loss
    "BREAKEVEN_TRIGGER_PCT": 0.018,  # Lock stop to breakeven once profit hits +1.8%
    
    # --- POSITION SIZING ---
    "BASE_RISK_LONG": 1.0,           # Uniform 1.0x long position
    "BASE_RISK_SHORT": 1.0,          # Uniform 1.0x short position
    "COMPRESSION_BOOST": 1.50,       # 1.5x position size during extreme compression (q <= 0.15)
    
    "EMA_SMOOTHING_SPAN": 18,        # 3-hour volatility forecast smoothing window
    "DEFAULT_MAKER_FEE": 0.00040     # 0.04% fee
}

# ==============================================================================
# 1. LIVE ENGINE PREPROCESSING LAYER (ROBUST CACHING + FULL FEATURES)
# ==============================================================================
class MarketDataStream:
    CACHE_FILENAME = "eth_usdt_5m_cache.csv"

    @staticmethod
    def _create_robust_session():
        session = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        session.mount("https://", HTTPAdapter(max_retries=retries))
        session.headers.update({"User-Agent": "Mozilla/5.0"})
        return session

    @classmethod
    def fetch_live_candles_paginated(cls, symbol, target_10m_bars):
        total_raw_needed = (target_10m_bars * 2) + 120
        
        # 1. CHECK LOCAL STORAGE FIRST
        if os.path.exists(cls.CACHE_FILENAME):
            print(f"[Data Stream] Found local cache ({cls.CACHE_FILENAME}). Verifying length...")
            try:
                df_cached = pd.read_csv(cls.CACHE_FILENAME, parse_dates=[0], index_col=0)
                df_cached.index.name = "timestamp"
                
                if len(df_cached) >= total_raw_needed:
                    print(f"[Data Stream] Local cache loaded successfully! (Skipping API download)")
                    df_cached = df_cached[~df_cached.index.duplicated(keep='first')]
                    return df_cached.iloc[-total_raw_needed:][["open", "high", "low", "close", "volume"]]
                else:
                    print(f"[Data Stream] Cache too small ({len(df_cached)} < {total_raw_needed}). Re-fetching...")
            except Exception as e:
                print(f"[Data Stream] Corrupted cache detected ({e}). Clearing local cache...")
                os.remove(cls.CACHE_FILENAME)

        # 2. FETCH FROM BINANCE (Robust Pagination)
        session = cls._create_robust_session()
        url = "https://api.binance.com/api/v3/klines"
        binance_limit_per_call = 1000
        chunks_needed = int(np.ceil(total_raw_needed / binance_limit_per_call))
        
        print(f"[Data Stream] Target 10m Bars: {target_10m_bars} | Requires {total_raw_needed} raw 5m candles.")
        print(f"[Data Stream] Segmenting requests into {chunks_needed} paginated API calls...")
        
        all_raw_candles = []
        end_time = None
        
        for chunk in range(chunks_needed):
            params = {"symbol": symbol, "interval": "5m", "limit": binance_limit_per_call}
            if end_time:
                params["endTime"] = end_time
                
            try:
                response = session.get(url, params=params, timeout=15)
                if response.status_code != 200:
                    time.sleep(2)
                    continue
                    
                chunk_data = response.json()
                if not chunk_data:
                    break
                    
                all_raw_candles = chunk_data + all_raw_candles
                end_time = chunk_data[0][0] - 1
                time.sleep(0.15)
                
            except requests.exceptions.RequestException:
                time.sleep(3)
                
        df = pd.DataFrame(all_raw_candles).iloc[:, :6]
        df.columns = ["open_time", "open", "high", "low", "close", "volume"]
        df[["open", "high", "low", "close", "volume"]] = df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric)
        df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
        df.set_index("timestamp", inplace=True)
        
        df = df[~df.index.duplicated(keep='first')]
        df_final = df[["open", "high", "low", "close", "volume"]]
        
        df_final.to_csv(cls.CACHE_FILENAME)
        print(f"[Data Stream] Saved {len(df_final)} raw 5m candles locally to {cls.CACHE_FILENAME}.")
        return df_final

    @staticmethod
    def resample_and_engineer(df_5m):
        rules = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
        df = df_5m.resample("10min", closed="left", label="left").agg(rules).dropna()
        
        df["log_returns"] = np.log(df["close"] / df["close"].shift(1))
        df["garman_klass"] = 0.5 * (np.log(df["high"] / df["low"]))**2 - (2*np.log(2)-1) * (np.log(df["close"] / df["open"]))**2
        df["log_range"] = np.log(df["high"] / df["low"])
        df["volume_change"] = df["volume"].pct_change()
        df["close_open_gap"] = np.log(df["open"] / df["close"].shift(1))
        
        denom = df["high"] - df["low"]
        denom = np.where(denom == 0, 1e-8, denom)
        df["norm_close_pos"] = (df["close"] - df["low"]) / denom
        
        return df.dropna()

# ==============================================================================
# 2. PURE HYBRID NN BREAKOUT BACKTEST ENGINE (RETURNS FULL EVALUATION DF)
# ==============================================================================
def run_pure_nn_breakout_backtest(df, raw_nn_predictions, channel_bars=60, fee_rate=0.0004, lookback_window=432, config=PURE_NN_CONFIG):
    evaluation_df = df.iloc[-len(raw_nn_predictions):].copy()
    
    # 1. SIGNAL SMOOTHING & VOLATILITY QUANTILES
    raw_pred_series = pd.Series(raw_nn_predictions, index=evaluation_df.index)
    evaluation_df["Smoothed_Predicted_Vol"] = raw_pred_series.ewm(
        span=config["EMA_SMOOTHING_SPAN"], adjust=False
    ).mean()
    
    evaluation_df['Pred_Vol_Quantile'] = evaluation_df['Smoothed_Predicted_Vol'].rolling(
        window=lookback_window
    ).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)
    
    # 24-Hour Volume Moving Average
    evaluation_df['Volume_SMA'] = evaluation_df['volume'].rolling(window=config["VOLUME_SMA_WINDOW"]).mean()
    
    # Extract arrays
    close = evaluation_df['close'].values if 'close' in evaluation_df.columns else evaluation_df['norm_close_pos'].values
    high = evaluation_df['high'].values
    low = evaluation_df['low'].values
    volume = evaluation_df['volume'].values
    vol_sma = evaluation_df['Volume_SMA'].values
    quantiles = evaluation_df['Pred_Vol_Quantile'].values
    pred_vols = evaluation_df['Smoothed_Predicted_Vol'].values
    macro_trend = evaluation_df['Macro_Trend'].values
    
    # Donchian Breakout Channels
    rolling_high = pd.Series(high, index=evaluation_df.index).shift(1).rolling(window=channel_bars).max().values
    rolling_low = pd.Series(low, index=evaluation_df.index).shift(1).rolling(window=channel_bars).min().values
    rolling_mid = (rolling_high + rolling_low) / 2.0
    
    n_bars = len(close)
    final_positions = np.zeros(n_bars)
    
    current_pos = 0.0
    entry_price = 0.0
    trade_tp_pct = 0.040
    bars_in_trade = 0
    cooldown_counter = 0
    breakeven_active = False
    
    trade_count = 0
    win_count = 0
    loss_count = 0
    
    # Execution Loop
    for i in range(n_bars):
        q = quantiles[i]
        curr_vol = pred_vols[i]
        curr_close = close[i]
        curr_macro = macro_trend[i]
        curr_vol_mult = volume[i] / vol_sma[i] if not np.isnan(vol_sma[i]) and vol_sma[i] > 0 else 1.0
        
        if current_pos != 0.0:
            bars_in_trade += 1
        else:
            bars_in_trade = 0
            if cooldown_counter > 0:
                cooldown_counter -= 1

        # --- ACTIVE POSITION RISK MANAGEMENT ---
        if current_pos != 0.0:
            unrealized_return = (curr_close - entry_price) / entry_price if current_pos > 0 else (entry_price - curr_close) / entry_price
            
            if unrealized_return >= config["BREAKEVEN_TRIGGER_PCT"]:
                breakeven_active = True

            effective_stop_pct = 0.000 if breakeven_active else -config["STOP_LOSS_PCT"]

            # Stop-Loss / Breakeven Exit
            if unrealized_return <= effective_stop_pct:
                if effective_stop_pct < 0:
                    loss_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue
                
            # Dynamic TP Exit
            if unrealized_return >= trade_tp_pct:
                win_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

            # Vol Expansion Peak Exit
            if q >= config["VOL_EXPANSION_EXIT_Q"] and unrealized_return > 0.01:
                win_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

            # Trailing Midpoint Exit
            if bars_in_trade >= config["MIN_HOLD_BARS"]:
                if (current_pos > 0 and curr_close < rolling_mid[i]) or (current_pos < 0 and curr_close > rolling_mid[i]):
                    if unrealized_return > 0:
                        win_count += 1
                    else:
                        loss_count += 1
                    current_pos = 0.0
                    trade_count += 1
                    cooldown_counter = config["COOLDOWN_BARS"]
                    final_positions[i] = 0.0
                    continue

            # Time Stop
            if bars_in_trade >= config["MAX_HOLD_BARS"]:
                if unrealized_return > 0:
                    win_count += 1
                else:
                    loss_count += 1
                current_pos = 0.0
                trade_count += 1
                cooldown_counter = config["COOLDOWN_BARS"]
                final_positions[i] = 0.0
                continue

        # --- NEW ENTRY EVALUATION ---
        if current_pos == 0.0 and cooldown_counter == 0 and not np.isnan(rolling_high[i]) and not np.isnan(rolling_low[i]):
            if q <= config["VOL_SQUEEZE_MAX_Q"] and curr_vol_mult >= config["VOLUME_MULT_THRESHOLD"]:
                size_multiplier = config["COMPRESSION_BOOST"] if q <= config["EXTREME_SQUEEZE_Q"] else 1.0
                vol_std = np.sqrt(max(1e-8, curr_vol))
                trade_tp_pct = max(config["MIN_TP_PCT"], config["DYNAMIC_TP_MULT"] * vol_std)
                
                # Long Breakout
                if curr_close > rolling_high[i] and curr_macro > 0:
                    current_pos = config["BASE_RISK_LONG"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False

                # Short Breakout
                elif curr_close < rolling_low[i] and curr_macro < 0:
                    current_pos = -config["BASE_RISK_SHORT"] * size_multiplier
                    entry_price = curr_close
                    bars_in_trade = 0
                    breakeven_active = False

        final_positions[i] = current_pos

    # Performance Accounting
    evaluation_df['Final_Position'] = final_positions
    evaluation_df['Asset_Return'] = evaluation_df['log_returns'] if 'log_returns' in evaluation_df.columns else evaluation_df['close'].pct_change().fillna(0.0)
    
    evaluation_df['Raw_Strategy_Return'] = evaluation_df['Final_Position'].shift(1).fillna(0.0) * evaluation_df['Asset_Return']
    evaluation_df['Position_Change'] = evaluation_df['Final_Position'].diff().abs().fillna(0.0)
    evaluation_df['Transaction_Fees'] = evaluation_df['Position_Change'] * fee_rate
    evaluation_df['Net_Strategy_Return'] = evaluation_df['Raw_Strategy_Return'] - evaluation_df['Transaction_Fees']
    
    total_net_return = evaluation_df['Net_Strategy_Return'].sum()
    total_fee_churn = evaluation_df['Transaction_Fees'].sum()
    
    cum_returns = (1.0 + evaluation_df['Net_Strategy_Return']).cumprod() if 'log_returns' not in evaluation_df.columns else evaluation_df['Net_Strategy_Return'].cumsum().apply(np.exp)
    running_max = cum_returns.cummax()
    max_drawdown = ((cum_returns - running_max) / running_max).min()
    
    win_rate = (win_count / trade_count * 100.0) if trade_count > 0 else 0.0
    
    # NOW INCLUDES evaluation_df RETURNED DIRECTLY
    return total_net_return, max_drawdown, total_fee_churn, trade_count, win_rate, evaluation_df


# ==============================================================================
# 3. METRICS CALCULATOR
# ==============================================================================
def compute_advanced_metrics(evaluation_df):
    net_returns = evaluation_df['Net_Strategy_Return']
    
    # Daily Returns & Annualized Sharpe Ratio (365 Crypto Days)
    daily_returns = net_returns.resample('D').sum()
    mean_daily = daily_returns.mean()
    std_daily = daily_returns.std()
    
    sharpe_ratio = (mean_daily / (std_daily + 1e-8)) * np.sqrt(365) if std_daily > 0 else 0.0
    
    # Profit Factor
    gross_profits = net_returns[net_returns > 0].sum()
    gross_losses = abs(net_returns[net_returns < 0].sum())
    profit_factor = (gross_profits / gross_losses) if gross_losses > 0 else np.inf
    
    # Monthly Return Breakdown
    monthly_returns = net_returns.groupby(pd.Grouper(freq='MS')).sum()
    
    return sharpe_ratio, profit_factor, monthly_returns


# ==============================================================================
# MAIN SYSTEM RUNNER (60-BAR OPTIMAL EVALUATION)
# ==============================================================================
if __name__ == "__main__":
    TEST_BARS = 25000  # 25,550 bars (~177 days)
    FEE_RATE = 0.0005   # 0.05% fee
    OPTIMAL_CHANNEL_BARS = 60  # 10 Hours (Optimal lookback)
    
    # Load Model Assets
    nn_model = tf.keras.models.load_model(MODEL_PATH, custom_objects={"stabilized_qlike_loss": stabilized_qlike_loss})
    nn_scaler = joblib.load(SCALER_PATH)
    wavelet = pywt.Wavelet(WAVELET_NAME)
    
    # 1. Fetch Shared Historical Target Data
    df_raw = MarketDataStream.fetch_live_candles_paginated(SYMBOL, target_10m_bars=TEST_BARS)
    df_features = MarketDataStream.resample_and_engineer(df_raw)
    
    # 2. Macro Trend Calculation
    print("[Strategy Engine] Computing 200-Hour EMA Macro Trend...")
    price_series = df_features["close"] if "close" in df_features.columns else df_features["norm_close_pos"]
    macro_ema_span = 1200
    df_features["Macro_EMA"] = price_series.ewm(span=macro_ema_span, adjust=False).mean()
    df_features["Macro_Trend"] = np.where(price_series >= df_features["Macro_EMA"], 1.0, -1.0)

    # 3. Extract Pure Hybrid NN Predictions
    print("[Strategy Engine] Extracting Pure Hybrid NN Volatility Predictions...")
    y_pred_nn, y_true = get_nn_predictions(df_features, TEST_BARS, nn_model, nn_scaler, wavelet, wavelet.dec_len)
    
    nn_qlike = np.mean(calculate_qlike(y_true, y_pred_nn))
    corr_nn = np.corrcoef(np.sqrt(y_true), np.sqrt(y_pred_nn))[0, 1]
    
    # 4. Run Backtest for Optimal 60-Bar Horizon & Capture evaluation_df
    print("[Strategy Engine] Running 60-Bar Pure Hybrid NN Breakout Strategy...")
    ret, dd, fees, trades, win_rate, eval_df = run_pure_nn_breakout_backtest(
        df_features, y_pred_nn, channel_bars=OPTIMAL_CHANNEL_BARS, fee_rate=FEE_RATE
    )
    
    # 5. Compute Advanced Performance Metrics
    sharpe_ratio, profit_factor, monthly_returns = compute_advanced_metrics(eval_df)

    # ==============================================================================
    # FINAL PRODUCTION PERFORMANCE REPORT
    # ==============================================================================
    print("\n" + "="*65)
    print("   OPTIMAL 60-BAR PURE HYBRID NN BREAKOUT PERFORMANCE REPORT")
    print("="*65)
    print(f"Evaluation Window Horizon   : {len(y_true)} (10m bars) (~177 Days)")
    print(f"Exchange Fee Assumed        : {FEE_RATE*100:.4f}%")
    print(f"Model Quality               : QLIKE Loss: {nn_qlike:.6f} | Vol Corr: {corr_nn:.4f}")
    print("-"*65)
    print("   EXECUTIVE PERFORMANCE SUMMARY")
    print("-"*65)
    print(f"Total Net Return            : {ret*100:+.2f}%")
    print(f"Max Peak-to-Trough Drawdown : {dd*100:.2f}%")
    print(f"Annualized Sharpe Ratio     : {sharpe_ratio:.2f}")
    print(f"Strategy Profit Factor      : {profit_factor:.2f}")
    print(f"Win Rate                    : {win_rate:.1f}%")
    print(f"Total Trades Executed       : {trades}")
    print(f"Total Exchange Fees Paid    : {fees*100:.2f}%")
    print("-"*65)
    print("   MONTH-BY-MONTH NET RETURN BREAKDOWN")
    print("-"*65)
    print(f"{'Month':<12} | {'Net Return':<12} | {'Status'}")
    print("-"*65)
    
    for month_dt, m_ret in monthly_returns.items():
        month_str = month_dt.strftime('%b %Y')
        status = "🟢 PROFIT" if m_ret > 0 else "🔴 LOSS"
        print(f"{month_str:<12} | {m_ret*100:>+10.2f}% | {status}")
        
    print("="*65)
    

[Data Stream] Found local cache (eth_usdt_5m_cache.csv). Verifying length...
[Data Stream] Local cache loaded successfully! (Skipping API download)
[Strategy Engine] Computing 200-Hour EMA Macro Trend...
[Strategy Engine] Extracting Pure Hybrid NN Volatility Predictions...
[Strategy Engine] Running 60-Bar Pure Hybrid NN Breakout Strategy...


C:\Users\dreal\AppData\Local\Temp\ipykernel_16304\3882892833.py:163: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ).apply(lambda x: (x[-1] >= x).mean() if len(x) > 0 else 0.5).fillna(0.5)



   OPTIMAL 60-BAR PURE HYBRID NN BREAKOUT PERFORMANCE REPORT
Evaluation Window Horizon   : 25000 (10m bars) (~177 Days)
Exchange Fee Assumed        : 0.0500%
Model Quality               : QLIKE Loss: 0.641256 | Vol Corr: 0.6208
-----------------------------------------------------------------
   EXECUTIVE PERFORMANCE SUMMARY
-----------------------------------------------------------------
Total Net Return            : +14.06%
Max Peak-to-Trough Drawdown : -10.00%
Annualized Sharpe Ratio     : 1.55
Strategy Profit Factor      : 1.08
Win Rate                    : 42.4%
Total Trades Executed       : 66
Total Exchange Fees Paid    : 8.22%
-----------------------------------------------------------------
   MONTH-BY-MONTH NET RETURN BREAKDOWN
-----------------------------------------------------------------
Month        | Net Return   | Status
-----------------------------------------------------------------
Feb 2026     |      -2.32% | 🔴 LOSS
Mar 2026     |     +13.74% | 🟢 PROFIT
Apr 202